# **DEFINISI FUNGSI MODULAR**

---



## Setup Fashion MNIST

In [1]:
def setup_fashion_mnist_environment(model_name):
    """
    Menangani mounting drive, setup path, dan inisialisasi DataLoaders.
    """
    from google.colab import drive
    import os
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    # 1. Mount Drive
    drive.mount('/content/drive', force_remount=True)

    # 2. Setup Paths
    model_dir = '/content/drive/MyDrive/model_modular'
    os.makedirs(model_dir, exist_ok=True)
    path = os.path.join(model_dir, model_name)

    # 3. DataLoaders
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_set = datasets.FashionMNIST(root='./data_fashion', train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(root='./data_fashion', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

    return path, train_loader, test_loader

print("✅ Helper function setup_fashion_mnist_environment siap digunakan.")

✅ Helper function setup_fashion_mnist_environment siap digunakan.


## Load or Training Model & Eval Model

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def evaluate_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def load_and_prepare_model(model_class, model_path, train_loader, test_loader, device, num_epochs=10, lr=0.001):
    model = model_class().to(device)
    history = {'loss': [], 'accuracy': []}

    if os.path.exists(model_path):
        print(f"✅ Loading existing model from {model_path}...")
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        history = checkpoint.get('history', history)
        print(f"✅ Model loaded. Previous accuracy: {history['accuracy'][-1]:.2f}%" if history['accuracy'] else "✅ Model loaded.")
    else:
        print(f"❌ No model found at {model_path}. Starting training...")
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0
            for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            acc = evaluate_accuracy(model, test_loader, device)
            avg_loss = running_loss / len(train_loader)
            history['loss'].append(avg_loss)
            history['accuracy'].append(acc)
            print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {acc:.2f}%")

        print(f"💾 Saving model to {model_path}...")
        torch.save({'model_state_dict': model.state_dict(), 'history': history}, model_path)
        print("✅ Training complete and model saved.")

    return model, history

print("✅ Unified Model Manager & Trainer defined!")

✅ Unified Model Manager & Trainer defined!


## Fungsi Kuantisasi Modular

Standard & Fine-Grained Quantization Ternary


In [3]:
import torch
import torch.nn as nn
import copy
import numpy as np

def standard_ternary_quantize(weight_tensor):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()
    abs_weights = torch.abs(flat_weights)
    if abs_weights.max().item() == 0:
        return torch.zeros_like(weight_tensor), 0.0
    delta = 0.7 * torch.mean(abs_weights)
    mask = abs_weights > delta
    if mask.sum() == 0:
        return torch.zeros_like(weight_tensor), 0.0
    alpha = abs_weights[mask].sum() / mask.sum().float()
    ternary_flat = torch.zeros_like(flat_weights)
    ternary_flat[flat_weights > delta] = alpha
    ternary_flat[flat_weights < -delta] = -alpha
    return ternary_flat.reshape(original_shape), alpha.item()

def apply_standard_ternary_to_model(model, verbose=False):
    model_q = copy.deepcopy(model)
    layer_alphas = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alpha = standard_ternary_quantize(module.weight.data)
            module.weight.data = ternary_weight
            layer_alphas.append(alpha)
            if verbose: print(f"  ✓ {name}: quantized with ̑={alpha:.4f}")
    return model_q, layer_alphas

def ternary_quantize_group(weight_tensor, group_size=4):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()
    n = flat_weights.numel()
    ternary_flat = torch.zeros_like(flat_weights)
    alphas = []
    for i in range(0, n, group_size):
        group = flat_weights[i:min(i+group_size, n)]
        if len(group) == 0: continue
        abs_group = torch.abs(group)
        if abs_group.max().item() == 0:
            alphas.append(0.0)
            continue
        thresholds = torch.linspace(abs_group.min().item(), abs_group.max().item(), steps=20)
        best_delta, best_score = 0, -float('inf')
        for delta in thresholds:
            mask = abs_group > delta
            if mask.sum() == 0: continue
            score = (abs_group[mask].sum() ** 2) / mask.sum().float()
            if score > best_score: best_score, best_delta = score, delta
        mask = abs_group > best_delta
        if mask.sum() == 0:
            alphas.append(0.0)
            continue
        alpha = abs_group[mask].sum() / mask.sum().float()
        alphas.append(alpha.item())
        ternary_group = torch.zeros_like(group)
        ternary_group[group > best_delta] = alpha
        ternary_group[group < -best_delta] = -alpha
        ternary_flat[i:min(i+group_size, n)] = ternary_group
    return ternary_flat.reshape(original_shape), alphas

def apply_fgq_to_model(model, group_size=4, verbose=False):
    if isinstance(group_size, (list, set, tuple)):
        results = {}
        for gs in sorted(list(group_size)):
            if verbose: print(f"\n--- Processing Group Size: {gs} ---")
            m_q, a_m = apply_fgq_to_model(model, gs, verbose)
            results[gs] = (m_q, a_m)
        return results

    model_q = copy.deepcopy(model)
    all_alpha_means = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alphas = ternary_quantize_group(module.weight.data, group_size)
            module.weight.data = ternary_weight
            m_alpha = np.mean(alphas) if alphas else 0
            all_alpha_means.append(m_alpha)
            if verbose: print(f"  ✓ {name}: FGQ N={group_size}, mean_̑={m_alpha:.4f}")
    return model_q, all_alpha_means

def find_best_fgq_model(fgq_results, test_loader, device):
    best_acc = -1.0
    best_gs = None
    best_model = None

    print("\n--- Evaluating All Group Sizes ---")
    for gs, (model_q, _) in fgq_results.items():
        # Import evaluate_accuracy here if not global, but it is expected from cell b1202b3e
        acc = evaluate_accuracy(model_q, test_loader, device)
        print(f"Group Size {gs}: Accuracy = {acc:.2f}%")
        if acc > best_acc:
            best_acc = acc
            best_gs = gs
            best_model = model_q

    print(f"\n✅ Best Result: Group Size {best_gs} with Accuracy {best_acc:.2f}%")
    return best_gs, best_model, best_acc

## Definisi Model

### LeNet-5

In [4]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Redefine Model Class
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = x.view(-1, 16 * 5 * 5)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 2. Re-initialize DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
test_dataset = datasets.FashionMNIST(root='./data_fashion', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 3. Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)

print("✅ Setup for verification complete: Model and Test Loader are ready.")

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 274kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.08MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.75MB/s]

✅ Setup for verification complete: Model and Test Loader are ready.


In [5]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Redefine Model Class for ResNet18
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18, self).__init__()
        # Load a pre-trained ResNet18 model
        self.model = models.resnet18(weights=None) # weights=None to avoid downloading ImageNet weights for now

        # Modify the first convolution layer for single-channel input (Fashion MNIST images are grayscale)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Modify the final fully connected layer to match the number of classes (Fashion MNIST has 10 classes)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

# Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet18().to(device)

print("✅ ResNet18 model defined and ready.")

✅ ResNet18 model defined and ready.


# **PENGUJIAN TRAINING & KUANTISASI**

---



## Model Setup

In [6]:
# 1. Siapkan environment (Drive, Path, Data)
model_path, train_loader, test_loader = setup_fashion_mnist_environment('resnet18_fashion_baseline.pth')

# 2. Panggil manager model
model_fp32, history = load_and_prepare_model(
    ResNet18,
    model_path,
    train_loader,
    test_loader,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history['accuracy'][-1]:.2f}%")

Mounted at /content/drive
❌ No model found at /content/drive/MyDrive/model_modular/resnet18_fashion_baseline.pth. Starting training...


Epoch 1/5: 100%|██████████| 469/469 [12:08<00:00,  1.55s/it]


Epoch 1: Loss = 0.4125, Accuracy = 87.69%


Epoch 2/5: 100%|██████████| 469/469 [12:23<00:00,  1.59s/it]


Epoch 2: Loss = 0.2932, Accuracy = 87.60%


...

Epoch 3: Loss = 0.2521, Accuracy = 89.59%


Epoch 4/5: 100%|██████████| 469/469 [11:51<00:00,  1.52s/it]


Epoch 4: Loss = 0.2303, Accuracy = 89.10%


Epoch 5/5: 100%|██████████| 469/469 [12:06<00:00,  1.55s/it]


Epoch 5: Loss = 0.2090, Accuracy = 89.91%
💾 Saving model to /content/drive/MyDrive/model_modular/resnet18_fashion_baseline.pth...
✅ Training complete and model saved.

Proses selesai. Akurasi Baseline: 89.91%


In [12]:
# 1. Siapkan environment (Drive, Path, Data)
model_path, train_loader, test_loader = setup_fashion_mnist_environment('lenet5_fashion_baseline.pth')

# 2. Panggil manager model
model_fp32_lenet, history_lenet = load_and_prepare_model(
    LeNet5,
    model_path,
    train_loader,
    test_loader,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history['accuracy'][-1]:.2f}%")

Mounted at /content/drive
✅ Loading existing model from /content/drive/MyDrive/model_modular/lenet5_fashion_baseline.pth...
✅ Model loaded. Previous accuracy: 86.17%

Proses selesai. Akurasi Baseline: 89.91%


## Kuantisasi Model

### *Standard Ternary Quantization*

In [7]:
model_ternary, alphas = apply_standard_ternary_to_model(model_fp32, verbose=True)
acc_ternary = evaluate_accuracy(model_ternary, test_loader, device)

print(f"Baseline: {history['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary:.2f}%")

  ✓ model.conv1: quantized with ̑=0.1096
  ✓ model.layer1.0.conv1: quantized with ̑=0.0827
  ✓ model.layer1.0.conv2: quantized with ̑=0.0822
  ✓ model.layer1.1.conv1: quantized with ̑=0.0815
  ✓ model.layer1.1.conv2: quantized with ̑=0.0812
  ✓ model.layer2.0.conv1: quantized with ̑=0.0621
  ✓ model.layer2.0.conv2: quantized with ̑=0.0607
  ✓ model.layer2.0.downsample.0: quantized with ̑=0.1542
  ✓ model.layer2.1.conv1: quantized with ̑=0.0593
  ✓ model.layer2.1.conv2: quantized with ̑=0.0579
  ✓ model.layer3.0.conv1: quantized with ̑=0.0431
  ✓ model.layer3.0.conv2: quantized with ̑=0.0401
  ✓ model.layer3.0.downsample.0: quantized with ̑=0.1116
  ✓ model.layer3.1.conv1: quantized with ̑=0.0372
  ✓ model.layer3.1.conv2: quantized with ̑=0.0373
  ✓ model.layer4.0.conv1: quantized with ̑=0.0284
  ✓ model.layer4.0.conv2: quantized with ̑=0.0259
  ✓ model.layer4.0.downsample.0: quantized with ̑=0.0787
  ✓ model.layer4.1.conv1: quantized with ̑=0.0251
  ✓ model.layer4.1.conv2: quantized wi

In [13]:
model_ternary_lenet, alphas_lenet = apply_standard_ternary_to_model(model_fp32_lenet, verbose=True)
acc_ternary_lenet = evaluate_accuracy(model_ternary_lenet, test_loader, device)

print(f"Baseline: {history['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")

  ✓ conv1: quantized with ̑=0.2242
  ✓ conv2: quantized with ̑=0.1076
  ✓ fc1: quantized with ̑=0.0722
  ✓ fc2: quantized with ̑=0.0842
  ✓ fc3: quantized with ̑=0.1235
Baseline: 89.91%
Ternary: 75.49%


### *Fine-Grained Ternary Quantization*

In [9]:
fgq_results = apply_fgq_to_model(model_fp32, group_size={4, 8, 16}, verbose=True)

best_gs, best_model_fgq, best_acc = find_best_fgq_model(fgq_results, test_loader, device)

print(f"\nBaseline Accuracy: {history['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs}: {best_acc:.2f}%")


--- Processing Group Size: 4 ---
  ✓ model.conv1: FGQ N=4, mean_̑=0.1095
  ✓ model.layer1.0.conv1: FGQ N=4, mean_̑=0.0845
  ✓ model.layer1.0.conv2: FGQ N=4, mean_̑=0.0840
  ✓ model.layer1.1.conv1: FGQ N=4, mean_̑=0.0838
  ✓ model.layer1.1.conv2: FGQ N=4, mean_̑=0.0833
  ✓ model.layer2.0.conv1: FGQ N=4, mean_̑=0.0639
  ✓ model.layer2.0.conv2: FGQ N=4, mean_̑=0.0623
  ✓ model.layer2.0.downsample.0: FGQ N=4, mean_̑=0.1603
  ✓ model.layer2.1.conv1: FGQ N=4, mean_̑=0.0610
  ✓ model.layer2.1.conv2: FGQ N=4, mean_̑=0.0594
  ✓ model.layer3.0.conv1: FGQ N=4, mean_̑=0.0443
  ✓ model.layer3.0.conv2: FGQ N=4, mean_̑=0.0413
  ✓ model.layer3.0.downsample.0: FGQ N=4, mean_̑=0.1148
  ✓ model.layer3.1.conv1: FGQ N=4, mean_̑=0.0383
  ✓ model.layer3.1.conv2: FGQ N=4, mean_̑=0.0384
  ✓ model.layer4.0.conv1: FGQ N=4, mean_̑=0.0295
  ✓ model.layer4.0.conv2: FGQ N=4, mean_̑=0.0269
  ✓ model.layer4.0.downsample.0: FGQ N=4, mean_̑=0.0811
  ✓ model.layer4.1.conv1: FGQ N=4, mean_̑=0.0259
  ✓ model.layer4.1.conv

In [16]:
fgq_results_lenet = apply_fgq_to_model(model_fp32_lenet, group_size={4, 8, 16}, verbose=True)

best_gs_lenet, best_model_fgq_lenet, best_acc_lenet = find_best_fgq_model(fgq_results_lenet, test_loader, device)

print(f"\nBaseline Accuracy: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs_lenet}: {best_acc_lenet:.2f}%")


--- Processing Group Size: 4 ---
  ✓ conv1: FGQ N=4, mean_̑=0.2248
  ✓ conv2: FGQ N=4, mean_̑=0.1052
  ✓ fc1: FGQ N=4, mean_̑=0.0727
  ✓ fc2: FGQ N=4, mean_̑=0.0868
  ✓ fc3: FGQ N=4, mean_̑=0.1297

--- Processing Group Size: 8 ---
  ✓ conv1: FGQ N=8, mean_̑=0.2171
  ✓ conv2: FGQ N=8, mean_̑=0.1064
  ✓ fc1: FGQ N=8, mean_̑=0.0758
  ✓ fc2: FGQ N=8, mean_̑=0.0883
  ✓ fc3: FGQ N=8, mean_̑=0.1334

--- Processing Group Size: 16 ---
  ✓ conv1: FGQ N=16, mean_̑=0.2158
  ✓ conv2: FGQ N=16, mean_̑=0.1070
  ✓ fc1: FGQ N=16, mean_̑=0.0771
  ✓ fc2: FGQ N=16, mean_̑=0.0864
  ✓ fc3: FGQ N=16, mean_̑=0.1286

--- Evaluating All Group Sizes ---
Group Size 4: Accuracy = 82.70%
Group Size 8: Accuracy = 80.74%
Group Size 16: Accuracy = 76.39%

✅ Best Result: Group Size 4 with Accuracy 82.70%

Baseline Accuracy: 86.17%
Akurasi Terbaik ditemukan pada Group Size 4: 82.70%


## Perbandingan Akurasi

### Lenet5

In [18]:
print(f"Baseline: {history['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary:.2f}%")
print(f"FGQ best Group Size: {best_acc:.2f}%")

Baseline: 89.91%
Ternary: 26.28%
FGQ best Group Size: 69.76%


### ResNet18

In [17]:
print(f"Baseline: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")
print(f"FGQ best Group Size: {best_acc_lenet:.2f}%")

Baseline: 86.17%
Ternary: 75.49%
FGQ best Group Size: 82.70%


# Ringkasan & Panduan Penggunaan Modular

Setelah melakukan refaktorisasi, eksperimen kuantisasi kini jauh lebih bersih dan terorganisir. Berikut adalah manfaat utama dari struktur baru ini:

1.  **Independensi Model**: Fungsi `apply_...` menggunakan `copy.deepcopy()`, sehingga model asli (FP32) tetap aman dan bisa digunakan berkali-kali untuk skenario berbeda.
2.  **Otomatisasi Training**: Fungsi `load_and_prepare_model` menangani logika pemuatan file dan training secara internal.
3.  **Kemudahan Eksperimen**: Membandingkan teknik kuantisasi kini hanya membutuhkan beberapa baris kode.

#### Contoh Penggunaan:

```python
# 1. Load/Train Baseline Model
model_fp32, history = load_and_prepare_model(LeNet5, model_path, train_loader, test_loader, device)

# 2. Jalankan Standard Ternary
model_ternary, alphas = apply_standard_ternary_to_model(model_fp32, verbose=True)
acc_ternary = evaluate_accuracy(model_ternary, test_loader, device)

# 3. Jalankan FGQ dengan N=4
model_fgq, mean_alphas = apply_fgq_to_model(model_fp32, group_size=4, verbose=True)
acc_fgq = evaluate_accuracy(model_fgq, test_loader, device)

print(f"Baseline: {history['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary:.2f}%")
print(f"FGQ (N=4): {acc_fgq:.2f}%")
```